# Step 9 — Folds 1-4 and ensemble

**Settings: `GPU T4 x2`, `Internet On`.**

**Inputs:** competition data, `rsna-knee-04-build-cache`, `rsna-knee-03-extract-all`, `rsna-knee-05-train-fold0`

## Why ensemble

You have one model trained on 80% of the data. Four more, each holding out a different fifth, then average all five predictions.

This works because the models make **different** mistakes — different initialisation, different training subset, different augmentation draws. Averaging cancels the independent errors while the shared signal survives. It is the most reliable gain in competitive ML, typically +0.01 to +0.02, and it needs no new ideas.

It also gives you an honest **out-of-fold** estimate: every training study gets a prediction from the one model that never saw it. That's a far better validation signal than the 58 gold studies, and it's what you'll use to tune blending later without touching the leaderboard.

## Cost and safety

~50 min per fold, ~3.5 hours total. Well inside the 12-hour commit limit.

Each fold saves its checkpoint as soon as it finishes, and the notebook **skips folds whose checkpoint already exists**. If it dies at fold 3, re-running resumes at fold 3 rather than starting over.

## Kept identical to fold 0

Same architecture, same 8 epochs, same seed, same per-label loss weighting. Changing anything here would make the folds incomparable and the ensemble incoherent.

Reference: fold 0 scored **val(pseudo) 0.785, gold(58) 0.803, public LB 0.826**.

### 1 · Setup

In [ ]:
import os, gc, glob, time, math, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

ROOT   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
CACHE  = glob.glob("/kaggle/input/notebooks/*/*/cache.npy")[0]
PSEUDO = glob.glob("/kaggle/input/notebooks/*/*/pseudo_labels.csv")[0]
GOLD   = glob.glob("/kaggle/input/notebooks/*/*/gold58.csv")[0]
FOLD0  = glob.glob("/kaggle/input/notebooks/*/*/model_fold0.pt")
print("cache:", CACHE, "\nfold0 ckpt:", FOLD0)

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
KEYS   = ["acl","mcl","medial_meniscus","lateral_meniscus","medial_oa","lateral_oa",
          "pf_oa","effusion","synovitis","bakers","contusion","fracture"]

arr    = np.load(CACHE, mmap_mode="r")
pseudo = pd.read_csv(PSEUDO)
gold   = pd.read_csv(GOLD)
train  = pd.read_csv(f"{ROOT}/train.csv")
N, NSER, NSL, PX, _ = arr.shape

assert (pseudo["StudyInstanceUID"].values == train["StudyInstanceUID"].values).all()
Y = pseudo[KEYS].values.astype(np.float32) / 100.0
nan_mask = np.isnan(Y).any(1)
Y = np.nan_to_num(Y, nan=0.2).astype(np.float32)
print(f"cache {arr.shape} | NaN-target studies {int(nan_mask.sum())}")
print("GPU:", torch.cuda.get_device_name(0))

### 2 · Splits and loss weights — identical to fold 0

In [ ]:
gold_ids = set(gold["StudyInstanceUID"])
is_gold  = train["StudyInstanceUID"].isin(gold_ids).values

t0 = time.time()
present = np.zeros((N, NSER), np.float32)
for i in range(N):
    for k in range(NSER):
        present[i, k] = float(arr[i, k, NSL // 2].max() > 0)
print(f"presence in {time.time()-t0:.0f}s | mean series/study {present.sum(1).mean():.2f}")

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    os.system("pip install -q iterative-stratification")
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

usable = np.where((~is_gold) & (present.sum(1) > 0) & (~nan_mask))[0]
Ybin   = (Y[usable] > 0.5).astype(int)
mskf   = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds  = np.zeros(len(usable), int)
for f, (_, v) in enumerate(mskf.split(usable, Ybin)):
    folds[v] = f
gd_idx = np.where(is_gold & (present.sum(1) > 0))[0]
print("fold sizes:", [int((folds == f).sum()) for f in range(5)], "| gold", len(gd_idx))

# per-label extraction reliability -> loss weights
gl_ids  = gold["StudyInstanceUID"].tolist()
Pg_ext  = np.nan_to_num(pseudo.set_index("StudyInstanceUID").loc[gl_ids, KEYS].values.astype(np.float32), nan=20.0)
Yg_true = gold.set_index("StudyInstanceUID").loc[gl_ids, LABELS].values.astype(int)
ext_auc = np.array([roc_auc_score(Yg_true[:, c], Pg_ext[:, c]) for c in range(12)])
w = ext_auc - 0.5; w = w / w.mean()
loss_w = torch.tensor(w, dtype=torch.float32).cuda()
print("loss weights:", np.round(w, 2))

### 3 · Dataset and model — unchanged

In [ ]:
class KneeDS(Dataset):
    def __init__(self, idx, train_mode):
        self.idx, self.train_mode = idx, train_mode
    def __len__(self): return len(self.idx)
    def __getitem__(self, n):
        i = self.idx[n]
        v = np.asarray(arr[i]).astype(np.float32) / 255.0
        if self.train_mode:
            ang   = np.random.uniform(-12, 12)
            scale = np.random.uniform(0.9, 1.1)
            tx    = np.random.uniform(-0.06, 0.06) * PX
            ty    = np.random.uniform(-0.06, 0.06) * PX
            M = cv2.getRotationMatrix2D((PX/2, PX/2), ang, scale)
            M[0, 2] += tx; M[1, 2] += ty
            gain = np.random.uniform(0.85, 1.15)
            bias = np.random.uniform(-0.08, 0.08)
            for s in range(NSER):
                if v[s].max() == 0: continue
                for j in range(NSL):
                    v[s, j] = cv2.warpAffine(v[s, j], M, (PX, PX), borderValue=0)
                v[s] = np.clip(v[s] * gain + bias, 0, 1)
        out = np.empty((NSER, NSL, 3, PX, PX), np.float32)
        for s in range(NSER):
            for j in range(NSL):
                lo, hi = max(j-1, 0), min(j+1, NSL-1)
                out[s, j] = np.stack([v[s, lo], v[s, j], v[s, hi]])
        return (torch.from_numpy(out), torch.from_numpy(present[i]), torch.from_numpy(Y[i]))

def loader(idx, train_mode, bs=6):
    return DataLoader(KneeDS(idx, train_mode), batch_size=bs, shuffle=train_mode,
                      num_workers=2, pin_memory=True, drop_last=train_mode,
                      persistent_workers=True)

import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        return (x * a.softmax(-1).unsqueeze(-1)).sum(1)

class Net(nn.Module):
    def __init__(self, backbone="efficientnet_b0", nlab=12, hid=256, pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))
    def forward(self, x, mask):
        B, S, T = x.shape[:3]
        f = self.enc(x.reshape(B * S * T, *x.shape[3:]))
        f = f.reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        return self.head(self.ser_pool(f, mask))

def macro_auc(P, T, thresh=0.5):
    P = np.nan_to_num(P, nan=0.5); out = []
    for c in range(T.shape[1]):
        y = (T[:, c] > thresh).astype(int)
        out.append(np.nan if y.min() == y.max() else roc_auc_score(y, P[:, c]))
    return np.nanmean(out), out

print("ready")

### 4 · Train folds 1-4

Each fold is a fresh model on a different 80/20 split. Checkpoints save immediately on completion, and existing checkpoints are skipped — so a crash costs one fold, not the run.

In [ ]:
EPOCHS, BS, LR = 8, 6, 3e-4

def train_fold(FOLD):
    ck_path = f"/kaggle/working/model_fold{FOLD}.pt"
    if os.path.exists(ck_path):
        print(f"fold {FOLD}: checkpoint exists, skipping"); return

    seed_all(42 + FOLD)
    tr_idx = usable[folds != FOLD]
    va_idx = usable[folds == FOLD]
    tl, vl, gl = loader(tr_idx, True), loader(va_idx, False), loader(gd_idx, False)

    model  = Net().cuda()
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=EPOCHS*len(tl), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda")
    lossf  = nn.BCEWithLogitsLoss(weight=loss_w)

    @torch.no_grad()
    def ev(dl):
        model.eval(); P, T = [], []
        for x, m, y in dl:
            with torch.amp.autocast("cuda"):
                p = model(x.cuda(non_blocking=True), m.cuda())
            P.append(p.float().sigmoid().cpu().numpy()); T.append(y.numpy())
        return np.concatenate(P), np.concatenate(T)

    print(f"\n{'='*60}\nFOLD {FOLD}  train {len(tr_idx):,} val {len(va_idx):,}\n{'='*60}", flush=True)
    best, best_state, best_pv = 0, None, None
    for ep in range(EPOCHS):
        model.train(); tot = 0; t0 = time.time()
        for bi, (x, m, y) in enumerate(tl):
            x, m, y = x.cuda(non_blocking=True), m.cuda(), y.cuda()
            with torch.amp.autocast("cuda"):
                loss = lossf(model(x, m), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            if not math.isfinite(loss.item()):
                print("non-finite loss, aborting fold"); return
            tot += loss.item()
        Pv, Tv = ev(vl); av, _ = macro_auc(Pv, Tv)
        Pg, Tg = ev(gl); ag, _ = macro_auc(Pg, Tg)
        print(f"  ep{ep} loss {tot/len(tl):.4f}  val {av:.4f}  gold {ag:.4f}  [{(time.time()-t0)/60:.0f}m]", flush=True)
        if av > best:
            best, best_pv = av, Pv
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save({"model": best_state, "fold": FOLD, "val_auc": av,
                        "gold_auc": ag, "epoch": ep}, ck_path)

    oof = pd.DataFrame(best_pv, columns=LABELS)
    oof.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values[va_idx])
    oof.to_csv(f"/kaggle/working/oof_fold{FOLD}.csv", index=False)
    print(f"fold {FOLD} done, best val {best:.4f}")
    del model, tl, vl, gl; gc.collect(); torch.cuda.empty_cache()


T0 = time.time()
for f in [1, 2, 3, 4]:
    train_fold(f)
print(f"\nall folds in {(time.time()-T0)/60:.0f} min")

### 5 · Ensemble on the gold holdout

Averages all five models' predictions on the 58 expert-labelled studies. Fold 0's checkpoint comes from the attached notebook 05 output.

What to look for: the ensemble should beat the mean of the individual folds. If it doesn't, the models are too correlated to be worth averaging — which would itself be informative.

In [ ]:
paths = [p for p in FOLD0] + [f"/kaggle/working/model_fold{f}.pt" for f in [1,2,3,4]]
paths = [p for p in paths if os.path.exists(p)]
print("ensembling", len(paths), "checkpoints")

gl = loader(gd_idx, False)
allP, singles = [], []
for p in paths:
    ck = torch.load(p, map_location="cpu", weights_only=False)
    m = Net(pretrained=False).cuda(); m.load_state_dict(ck["model"]); m.eval()
    P, T = [], []
    with torch.no_grad():
        for x, mk, y in gl:
            with torch.amp.autocast("cuda"):
                o = m(x.cuda(non_blocking=True), mk.cuda())
            P.append(o.float().sigmoid().cpu().numpy()); T.append(y.numpy())
    P, T = np.concatenate(P), np.concatenate(T)
    a, _ = macro_auc(P, T); singles.append(a); allP.append(P)
    print(f"  fold {ck['fold']}: gold {a:.4f}")
    del m; gc.collect(); torch.cuda.empty_cache()

ens = np.mean(allP, 0)
ae, per = macro_auc(ens, T)
print(f"\nmean of singles : {np.mean(singles):.4f}")
print(f"ENSEMBLE (n={len(allP)}): {ae:.4f}   ({ae - np.mean(singles):+.4f})")
print(f"fold 0 alone reference: 0.8033\n")
print("per-label, ensemble on gold:")
for l, a in zip(LABELS, per):
    print(f"  {l:<18} {a:.3f}")

### 6 · Combined out-of-fold predictions

Every training study now has a prediction from the one model that never saw it. This becomes your offline tuning set — far more reliable than 58 studies, and it costs no submissions.

In [ ]:
oofs = sorted(glob.glob("/kaggle/working/oof_fold*.csv")) + \
       [p for p in glob.glob("/kaggle/input/notebooks/*/*/oof_fold0.csv")]
if oofs:
    allo = pd.concat([pd.read_csv(p) for p in oofs], ignore_index=True)
    allo = allo.drop_duplicates("StudyInstanceUID")
    allo.to_csv("/kaggle/working/oof_all.csv", index=False)
    print(f"combined OOF: {len(allo):,} studies from {len(oofs)} folds")

    truth = pseudo.set_index("StudyInstanceUID").loc[allo.StudyInstanceUID, KEYS].values / 100.0
    ao, _ = macro_auc(allo[LABELS].values, np.nan_to_num(truth, nan=0.2))
    print(f"OOF macro AUC vs pseudo-labels: {ao:.4f}")